## Acknowledgement

This notebook is heavily inspired by the excellent notebook:

[PS S6E5 RealMLP PyTabKit by yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e5-realmlp-pytabkit)

That notebook provided a very strong RealMLP baseline and was especially helpful for understanding effective PyTabKit / RealMLP settings for this competition.  
In particular, the RealMLP parameter choices such as the network shape, activation, PLR-related settings, label smoothing, dropout schedule, and transformation pipeline were used as an important reference.

Many thanks to yekenot for sharing such a strong and well-structured notebook.

In [ ]:
# ============================================================
# Strong RealMLP-TD Pipeline
# Good-parts version:
# - no digit features
# - ratio features
# - floor / round / quantile-bin categorical features
# - global count encoding
# - fold-wise frequency encoding
# - fold-wise target encoding
# - TE row-wise statistics
# - orig augmentation
# - strong RealMLP params
# ============================================================

!pip install -q pytabkit

import os
import gc
import random
import warnings
from itertools import combinations
from importlib.metadata import version

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import KBinsDiscretizer

import torch

try:
    from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier
except Exception:
    from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings("ignore")


# ============================================================
# Config
# ============================================================

class CFG:
    SEED = 42
    FOLDS = 5

    ID = "id"
    TARGET = "PitNextLap"

    DEVICE = "cuda"
    USE_ORIG = True
    ORIG_MODE = "all"      # "all" or "fold"

    USE_CUML_TE = True
    TE_FOLDS = 5
    TE_SMOOTH = 20

    BATCH_SIZE = 512
    PREDICT_BATCH_SIZE = 8192

    VAL_METRIC_NAME = "1-auc_ovr"
    VERBOSITY = 2

    OUTPUT_PREFIX = "strong_realmlp_goodparts"


# ============================================================
# Optional cuML TargetEncoder
# ============================================================

HAS_CUML = False

if CFG.USE_CUML_TE:
    try:
        import cudf
        from cuml.preprocessing import TargetEncoder as cuTargetEncoder
        HAS_CUML = True
        print("cuML TargetEncoder: available")
    except Exception as e:
        HAS_CUML = False
        print("cuML TargetEncoder: unavailable -> use manual TE fallback")
        print(repr(e))


# ============================================================
# Utils
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_colname(c):
    return (
        str(c)
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("/", "_div_")
        .replace("-", "_")
        .replace(".", "p")
    )


def round_to_step(s, step):
    return np.round(pd.to_numeric(s, errors="coerce") / step) * step


def make_key(df, cols):
    cols = list(cols)
    key = df[cols[0]].astype(str).fillna("__nan__")
    for c in cols[1:]:
        key = key + "__" + df[c].astype(str).fillna("__nan__")
    return key


def to_numpy(x):
    if hasattr(x, "get"):
        return x.get()
    return np.asarray(x)


def make_inner_fold_ids(y, n_splits=5, seed=42):
    fold_ids = np.zeros(len(y), dtype=np.int32)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    dummy = np.zeros((len(y), 1))

    for fold, (_, va_idx) in enumerate(skf.split(dummy, y)):
        fold_ids[va_idx] = fold

    return fold_ids


def get_proba(model, X):
    pred = model.predict_proba(X)
    pred = np.asarray(pred)

    if pred.ndim == 2 and pred.shape[1] == 2:
        return pred[:, 1].astype(np.float32)

    if pred.ndim == 2 and pred.shape[1] == 1:
        return pred[:, 0].astype(np.float32)

    return pred.reshape(-1).astype(np.float32)


seed_everything(CFG.SEED)

print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))
print("Device:", CFG.DEVICE)


# ============================================================
# Load Data
# ============================================================

train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e5/test.csv")
orig = pd.read_csv(
    "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
)

if "Normalized_TyreLife" in orig.columns:
    orig = orig.drop(columns=["Normalized_TyreLife"])

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Orig shape :", orig.shape)


train_id = train[CFG.ID].copy()
test_id = test[CFG.ID].copy()

y = train[CFG.TARGET].astype(int).reset_index(drop=True)
y_orig = orig[CFG.TARGET].astype(int).reset_index(drop=True)

X = train.drop(columns=[CFG.ID, CFG.TARGET]).reset_index(drop=True)
X_test = test.drop(columns=[CFG.ID]).reset_index(drop=True)
X_orig = orig.drop(columns=[CFG.TARGET]).reset_index(drop=True)

del train, test, orig
gc.collect()

BASE_COLS = list(X.columns)
ORIG_CAT_COLS = X.select_dtypes(include=["object"]).columns.tolist()
ORIG_NUM_COLS = [c for c in BASE_COLS if c not in ORIG_CAT_COLS]

print("Base columns:", len(BASE_COLS))
print("Original categorical columns:", ORIG_CAT_COLS)
print("Original numerical columns:", len(ORIG_NUM_COLS))


# ============================================================
# Global Feature Engineering
# ============================================================

CAT_LIKE_COLS = []


def add_col_to_cat_like(c):
    if c not in CAT_LIKE_COLS:
        CAT_LIKE_COLS.append(c)


def add_basic_features(dfs):
    for df in dfs:
        df["_LapNumber_div_RaceProgress"] = (
            df["LapNumber"] / (pd.to_numeric(df["RaceProgress"], errors="coerce") + 1e-6)
        ).astype("float32")

        df["_TyreLife_div_LapNumber"] = (
            df["TyreLife"] / pd.to_numeric(df["LapNumber"], errors="coerce").clip(lower=1)
        ).astype("float32")

        df["_TyreLife_minus_LapNumber"] = (
            pd.to_numeric(df["TyreLife"], errors="coerce")
            - pd.to_numeric(df["LapNumber"], errors="coerce")
        ).astype("float32")

        df["_LapNumber_x_RaceProgress"] = (
            pd.to_numeric(df["LapNumber"], errors="coerce")
            * pd.to_numeric(df["RaceProgress"], errors="coerce")
        ).astype("float32")


def add_num_as_cat_features(dfs):
    # exact / rounded / step cats for key continuous columns
    key_cont_cols = [
        "LapTime (s)",
        "LapTime_Delta",
        "Cumulative_Degradation",
    ]

    round_config = {
        "LapTime (s)": {
            "round_digits": [1, 0],
            "round_steps": [0.5, 1.0, 2.0, 5.0],
        },
        "LapTime_Delta": {
            "round_digits": [1, 0],
            "round_steps": [0.5, 1.0, 2.0, 5.0, 10.0],
        },
        "Cumulative_Degradation": {
            "round_digits": [1, 0],
            "round_steps": [1.0, 2.0, 5.0, 10.0, 20.0],
        },
    }

    for c in key_cont_cols:
        if c not in dfs[0].columns:
            continue

        sc = safe_colname(c)

        exact_col = f"{sc}_exact_cat"
        for df in dfs:
            df[exact_col] = pd.to_numeric(df[c], errors="coerce").round(3).astype(str)
        add_col_to_cat_like(exact_col)

        cfg = round_config[c]

        for d in cfg["round_digits"]:
            nc = f"{sc}_round{d}_cat"
            for df in dfs:
                df[nc] = pd.to_numeric(df[c], errors="coerce").round(d).astype(str)
            add_col_to_cat_like(nc)

        for step in cfg["round_steps"]:
            step_name = str(step).replace(".", "p")
            nc = f"{sc}_step_{step_name}_cat"
            for df in dfs:
                df[nc] = round_to_step(df[c], step).astype(str)
            add_col_to_cat_like(nc)

    # floor cats for all numerical-ish columns
    floor_base = list(dict.fromkeys(
        ORIG_NUM_COLS
        + [
            "_LapNumber_div_RaceProgress",
            "_TyreLife_div_LapNumber",
            "_TyreLife_minus_LapNumber",
            "_LapNumber_x_RaceProgress",
        ]
    ))

    for c in floor_base:
        if c not in dfs[0].columns:
            continue

        nc = f"{safe_colname(c)}_floor_cat"
        for df in dfs:
            df[nc] = np.floor(pd.to_numeric(df[c], errors="coerce")).fillna(-999).astype("int32").astype(str)
        add_col_to_cat_like(nc)


def add_quantile_bins(dfs):
    # Fit bins on train + orig + test because this is unsupervised.
    bin_config = {
        "RaceProgress": [50, 100, 200],
        "LapTime (s)": [7, 15, 30],
        "TyreLife": [20, 50],
        "LapNumber": [20, 50],
        "_LapNumber_div_RaceProgress": [50],
        "_TyreLife_div_LapNumber": [50],
    }

    ref = pd.concat(dfs, axis=0, ignore_index=True)

    for c, bins_list in bin_config.items():
        if c not in ref.columns:
            continue

        values = pd.to_numeric(ref[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

        for n_bins in bins_list:
            nc = f"{safe_colname(c)}_q{n_bins}_bin_cat"

            kb = KBinsDiscretizer(
                n_bins=n_bins,
                encode="ordinal",
                strategy="quantile",
                subsample=None,
            )

            kb.fit(values.values.reshape(-1, 1))

            for df in dfs:
                x = (
                    pd.to_numeric(df[c], errors="coerce")
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0)
                    .values
                    .reshape(-1, 1)
                )
                df[nc] = kb.transform(x).ravel().astype("int32").astype(str)

            add_col_to_cat_like(nc)


def add_interaction_cat_features(dfs):
    # Small but strong interaction categories from the simpler RealMLP code
    important_combos = [
        ("Race", "Compound"),
        ("Race", "Year"),
        ("Driver", "Compound"),
        ("Driver", "Race"),
        ("Compound", "LapNumber"),
        ("Race", "LapNumber"),
        ("LapNumber", "TyreLife"),
        ("Position", "RaceProgress"),
    ]

    combo_cols = []

    for cols in important_combos:
        if not all(c in dfs[0].columns for c in cols):
            continue

        nc = "cat2__" + "__".join([safe_colname(c) for c in cols])
        combo_cols.append(nc)

        for df in dfs:
            df[nc] = make_key(df, cols).astype(str)

        add_col_to_cat_like(nc)

    return combo_cols


def add_global_count_features(dfs):
    # Count features are target-free, so fit on train + orig.
    train_df, test_df, orig_df = dfs

    count_cols = (
        ORIG_CAT_COLS
        + [
            "Year_floor_cat",
            "PitStop_floor_cat",
            "LapNumber_floor_cat",
            "TyreLife_floor_cat",
            "Position_floor_cat",
            "RaceProgress_q200_bin_cat",
            "LapTime_s_q15_bin_cat",
        ]
        + [c for c in CAT_LIKE_COLS if c.startswith("cat2__")]
    )

    count_cols = [c for c in list(dict.fromkeys(count_cols)) if c in train_df.columns]

    ref = pd.concat([train_df[count_cols], orig_df[count_cols]], axis=0, ignore_index=True)

    for c in count_cols:
        nc = f"cnt__{safe_colname(c)}"
        vc = ref[c].astype(str).value_counts(dropna=False)

        for df in dfs:
            df[nc] = df[c].astype(str).map(vc).fillna(0).astype("float32")


# original categorical columns as string
for c in ORIG_CAT_COLS:
    for df in [X, X_test, X_orig]:
        df[c] = df[c].astype(str).fillna("__nan__")
    add_col_to_cat_like(c)

add_basic_features([X, X_test, X_orig])
add_num_as_cat_features([X, X_test, X_orig])
add_quantile_bins([X, X_test, X_orig])
COMBO_CAT_COLS = add_interaction_cat_features([X, X_test, X_orig])
add_global_count_features([X, X_test, X_orig])

print("X shape after global FE     :", X.shape)
print("X_test shape after global FE:", X_test.shape)
print("X_orig shape after global FE:", X_orig.shape)
print("CAT_LIKE_COLS:", len(CAT_LIKE_COLS))
print("COMBO_CAT_COLS:", COMBO_CAT_COLS)


# ============================================================
# Fold-wise FE / TE
# ============================================================

TE_BASE = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "RaceProgress",
    "Position_Change",
]

SELECTED_PAIR_SPECS = [
    ("Driver", "Compound"),
    ("Driver", "Race"),
    ("Driver", "LapNumber"),
    ("Driver", "TyreLife"),
    ("Compound", "LapNumber"),
    ("Compound", "TyreLife"),
    ("Race", "LapNumber"),
    ("Race", "Stint"),
    ("LapNumber", "Stint"),
    ("LapNumber", "TyreLife"),
    ("Position", "LapNumber"),
    ("Position", "RaceProgress"),
    ("RaceProgress", "TyreLife"),
    ("Position_Change", "LapNumber"),
    ("Position_Change", "TyreLife"),
]

SELECTED_PAIR_SPECS = [
    cols for cols in SELECTED_PAIR_SPECS
    if all(c in X.columns for c in cols)
]

FE_SINGLE_COLS = list(dict.fromkeys(
    ORIG_CAT_COLS
    + [
        "Year_floor_cat",
        "PitStop_floor_cat",
        "LapNumber_floor_cat",
        "TyreLife_floor_cat",
        "Position_floor_cat",
        "RaceProgress_q200_bin_cat",
        "LapTime_s_q15_bin_cat",
        "LapTime_Delta_round0_cat",
        "Cumulative_Degradation_round0_cat",
    ]
    + COMBO_CAT_COLS
))

FE_SINGLE_COLS = [c for c in FE_SINGLE_COLS if c in X.columns]

TE_SINGLE_COLS = list(dict.fromkeys(
    ORIG_CAT_COLS
    + [
        "Year_floor_cat",
        "PitStop_floor_cat",
        "LapNumber_floor_cat",
        "TyreLife_floor_cat",
        "Position_floor_cat",
        "RaceProgress_q200_bin_cat",
        "LapTime_s_q15_bin_cat",
        "LapTime_Delta_round0_cat",
        "Cumulative_Degradation_round0_cat",
    ]
    + COMBO_CAT_COLS
))

TE_SINGLE_COLS = [c for c in TE_SINGLE_COLS if c in X.columns]

print("FE_SINGLE_COLS:", len(FE_SINGLE_COLS))
print("TE_SINGLE_COLS:", len(TE_SINGLE_COLS))
print("SELECTED_PAIR_SPECS:", len(SELECTED_PAIR_SPECS))


def add_frequency_encode(X_tr, X_val, X_tst, cols, out_col, normalize=True):
    cols = list(cols)

    tr_key = make_key(X_tr, cols)
    val_key = make_key(X_val, cols)
    tst_key = make_key(X_tst, cols)

    vc = tr_key.value_counts(dropna=False)
    denom = len(X_tr) if normalize else 1.0

    X_tr[out_col] = (tr_key.map(vc).fillna(0).astype("float32").values / denom).astype("float32")
    X_val[out_col] = (val_key.map(vc).fillna(0).astype("float32").values / denom).astype("float32")
    X_tst[out_col] = (tst_key.map(vc).fillna(0).astype("float32").values / denom).astype("float32")


def add_manual_target_encode(
    X_tr,
    X_val,
    X_tst,
    y_tr,
    cols,
    out_col,
    n_folds=5,
    smooth=20,
    seed=42,
):
    cols = list(cols)
    y_arr = np.asarray(y_tr).astype(np.float32)
    prior = float(np.mean(y_arr))

    tr_key_full = make_key(X_tr, cols)
    val_key = make_key(X_val, cols)
    tst_key = make_key(X_tst, cols)

    oof = np.zeros(len(X_tr), dtype=np.float32)

    inner = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for in_idx, hold_idx in inner.split(np.zeros((len(X_tr), 1)), y_arr):
        key_in = tr_key_full.iloc[in_idx]
        y_in = y_arr[in_idx]

        tmp = pd.DataFrame({"key": key_in.values, "y": y_in})
        stats = tmp.groupby("key")["y"].agg(["mean", "count"])

        enc = (
            (stats["mean"] * stats["count"] + prior * smooth)
            / (stats["count"] + smooth)
        )

        key_hold = tr_key_full.iloc[hold_idx]
        oof[hold_idx] = key_hold.map(enc).fillna(prior).astype("float32").values

    tmp_full = pd.DataFrame({"key": tr_key_full.values, "y": y_arr})
    stats_full = tmp_full.groupby("key")["y"].agg(["mean", "count"])

    enc_full = (
        (stats_full["mean"] * stats_full["count"] + prior * smooth)
        / (stats_full["count"] + smooth)
    )

    X_tr[out_col] = oof
    X_val[out_col] = val_key.map(enc_full).fillna(prior).astype("float32").values
    X_tst[out_col] = tst_key.map(enc_full).fillna(prior).astype("float32").values


def add_cuml_target_encode(
    X_tr,
    X_val,
    X_tst,
    y_tr,
    cols,
    out_col,
    fold_ids,
    seed=42,
    smooth=20,
    n_folds=5,
):
    cols = list(cols)

    te = cuTargetEncoder(
        n_folds=n_folds,
        smooth=smooth,
        seed=seed,
        split_method="random",
        output_type="numpy",
        stat="mean",
        multi_feature_mode="combination",
    )

    X_tr_g = cudf.from_pandas(X_tr[cols].astype(str))
    X_val_g = cudf.from_pandas(X_val[cols].astype(str))
    X_tst_g = cudf.from_pandas(X_tst[cols].astype(str))
    y_tr_g = cudf.Series(np.asarray(y_tr).astype(np.float32))
    fold_ids_g = cudf.Series(fold_ids.astype(np.int32))

    tr_enc = te.fit_transform(X_tr_g, y_tr_g, fold_ids=fold_ids_g)
    val_enc = te.transform(X_val_g)
    tst_enc = te.transform(X_tst_g)

    X_tr[out_col] = to_numpy(tr_enc).reshape(-1).astype("float32")
    X_val[out_col] = to_numpy(val_enc).reshape(-1).astype("float32")
    X_tst[out_col] = to_numpy(tst_enc).reshape(-1).astype("float32")

    del X_tr_g, X_val_g, X_tst_g, y_tr_g, fold_ids_g, te
    gc.collect()


def add_target_encode(
    X_tr,
    X_val,
    X_tst,
    y_tr,
    cols,
    out_col,
    fold_ids,
    seed=42,
):
    if HAS_CUML:
        add_cuml_target_encode(
            X_tr=X_tr,
            X_val=X_val,
            X_tst=X_tst,
            y_tr=y_tr,
            cols=cols,
            out_col=out_col,
            fold_ids=fold_ids,
            seed=seed,
            smooth=CFG.TE_SMOOTH,
            n_folds=CFG.TE_FOLDS,
        )
    else:
        add_manual_target_encode(
            X_tr=X_tr,
            X_val=X_val,
            X_tst=X_tst,
            y_tr=y_tr,
            cols=cols,
            out_col=out_col,
            n_folds=CFG.TE_FOLDS,
            smooth=CFG.TE_SMOOTH,
            seed=seed,
        )


def add_te_rowwise_stats(X_tr, X_val, X_tst):
    te_cols = [
        c for c in X_tr.columns
        if c.startswith("te__") or c.startswith("te2__")
    ]

    if len(te_cols) == 0:
        return []

    stat_cols = []

    for df in [X_tr, X_val, X_tst]:
        arr = df[te_cols].astype("float32")

        df["te_stat_mean"] = arr.mean(axis=1).astype("float32")
        df["te_stat_std"] = arr.std(axis=1).fillna(0).astype("float32")
        df["te_stat_min"] = arr.min(axis=1).astype("float32")
        df["te_stat_max"] = arr.max(axis=1).astype("float32")
        df["te_stat_range"] = (df["te_stat_max"] - df["te_stat_min"]).astype("float32")

    stat_cols = [
        "te_stat_mean",
        "te_stat_std",
        "te_stat_min",
        "te_stat_max",
        "te_stat_range",
    ]

    return stat_cols


# ============================================================
# RealMLP preprocessing
# ============================================================

def prepare_realmlp_frames(X_tr, X_val, X_tst, cat_like_cols):
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    X_tst = X_tst.copy()

    cat_like = set(cat_like_cols)
    cat_like.update([c for c in X_tr.columns if c.startswith("cat2__")])
    cat_like.update([c for c in X_tr.columns if c.endswith("_cat")])
    cat_like.update([c for c in X_tr.columns if c.endswith("_bin_cat")])

    for c in X_tr.columns:
        if c in cat_like or X_tr[c].dtype == "object":
            X_tr[c] = X_tr[c].astype(str).fillna("__nan__")
            X_val[c] = X_val[c].astype(str).fillna("__nan__")
            X_tst[c] = X_tst[c].astype(str).fillna("__nan__")
        else:
            X_tr[c] = pd.to_numeric(X_tr[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            X_val[c] = pd.to_numeric(X_val[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            X_tst[c] = pd.to_numeric(X_tst[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

            med = X_tr[c].median()
            if not np.isfinite(med):
                med = 0.0

            X_tr[c] = X_tr[c].fillna(med).astype("float32")
            X_val[c] = X_val[c].fillna(med).astype("float32")
            X_tst[c] = X_tst[c].fillna(med).astype("float32")

    return X_tr, X_val, X_tst


# ============================================================
# RealMLP configs
# ============================================================

REALMLP_CONFIGS = [
    {
        "name": "silu_plr_512x256x128_ens12",
        "params": {
            "n_ens": 12,
            "n_epochs": 8,
            "batch_size": CFG.BATCH_SIZE,
            "predict_batch_size": CFG.PREDICT_BATCH_SIZE,

            "use_early_stopping": True,
            "early_stopping_additive_patience": 12,
            "early_stopping_multiplicative_patience": 1,

            "lr": 0.03,
            "wd": 0.018,
            "sq_mom": 0.98,
            "lr_sched": "lin_cos_log_15",
            "first_layer_lr_factor": 0.25,

            "embedding_size": 6,
            "max_one_hot_cat_size": 18,
            "hidden_sizes": [512, 256, 128],
            "act": "silu",
            "p_drop": 0.05,
            "p_drop_sched": "expm4t",

            "plr_hidden_1": 16,
            "plr_hidden_2": 8,
            "plr_act_name": "gelu",
            "plr_lr_factor": 0.1151,
            "plr_sigma": 2.33,

            "ls_eps": 0.01,
            "ls_eps_sched": "sqrt_cos",

            "add_front_scale": False,
            "bias_init_mode": "neg-uniform-dynamic-2",
            "tfms": [
                "one_hot",
                "median_center",
                "robust_scale",
                "smooth_clip",
                "embedding",
                "l2_normalize",
            ],
        },
    },
    {
        "name": "silu_plr_768x384x192_ens8",
        "params": {
            "n_ens": 8,
            "n_epochs": 8,
            "batch_size": CFG.BATCH_SIZE,
            "predict_batch_size": CFG.PREDICT_BATCH_SIZE,

            "use_early_stopping": True,
            "early_stopping_additive_patience": 12,
            "early_stopping_multiplicative_patience": 1,

            "lr": 0.025,
            "wd": 0.02,
            "sq_mom": 0.98,
            "lr_sched": "lin_cos_log_15",
            "first_layer_lr_factor": 0.25,

            "embedding_size": 6,
            "max_one_hot_cat_size": 18,
            "hidden_sizes": [768, 384, 192],
            "act": "silu",
            "p_drop": 0.07,
            "p_drop_sched": "expm4t",

            "plr_hidden_1": 16,
            "plr_hidden_2": 8,
            "plr_act_name": "gelu",
            "plr_lr_factor": 0.1151,
            "plr_sigma": 2.33,

            "ls_eps": 0.01,
            "ls_eps_sched": "sqrt_cos",

            "add_front_scale": False,
            "bias_init_mode": "neg-uniform-dynamic-2",
            "tfms": [
                "one_hot",
                "median_center",
                "robust_scale",
                "smooth_clip",
                "embedding",
                "l2_normalize",
            ],
        },
    },
]

print("RealMLP configs:")
for cfg in REALMLP_CONFIGS:
    print(" -", cfg["name"], cfg["params"]["hidden_sizes"], "n_ens=", cfg["params"]["n_ens"])


def train_one_realmlp(X_tr, y_tr, X_val, y_val, X_tst, cfg, seed):
    params = dict(cfg["params"])

    params.update({
        "random_state": seed,
        "device": CFG.DEVICE,
        "verbosity": CFG.VERBOSITY,
        "val_metric_name": CFG.VAL_METRIC_NAME,
    })

    model = RealMLP_TD_Classifier(**params)

    # external validation is useful for RealMLP early stopping / model selection
    model.fit(X_tr, y_tr, X_val, y_val)

    val_pred = get_proba(model, X_val)
    tst_pred = get_proba(model, X_tst)

    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_pred, tst_pred


# ============================================================
# CV Training
# ============================================================

skf = StratifiedKFold(
    n_splits=CFG.FOLDS,
    shuffle=True,
    random_state=CFG.SEED,
)

if CFG.ORIG_MODE == "fold":
    orig_skf = StratifiedKFold(
        n_splits=CFG.FOLDS,
        shuffle=True,
        random_state=CFG.SEED,
    )
    orig_splits = list(orig_skf.split(X_orig, y_orig))
else:
    orig_splits = [None] * CFG.FOLDS

oof_dict = {
    cfg["name"]: np.zeros(len(X), dtype=np.float32)
    for cfg in REALMLP_CONFIGS
}

test_dict = {
    cfg["name"]: np.zeros(len(X_test), dtype=np.float32)
    for cfg in REALMLP_CONFIGS
}

fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}/{CFG.FOLDS}")
    print("=" * 80)

    X_tr = X.iloc[tr_idx].copy().reset_index(drop=True)
    y_tr = y.iloc[tr_idx].copy().reset_index(drop=True)

    X_val = X.iloc[val_idx].copy().reset_index(drop=True)
    y_val = y.iloc[val_idx].copy().reset_index(drop=True)

    X_tst = X_test.copy().reset_index(drop=True)

    if CFG.USE_ORIG:
        if CFG.ORIG_MODE == "fold":
            or_tr_idx, _ = orig_splits[fold - 1]
            X_or = X_orig.iloc[or_tr_idx].copy().reset_index(drop=True)
            y_or = y_orig.iloc[or_tr_idx].copy().reset_index(drop=True)
        else:
            X_or = X_orig.copy().reset_index(drop=True)
            y_or = y_orig.copy().reset_index(drop=True)

        X_tr = pd.concat([X_tr, X_or], axis=0, ignore_index=True)
        y_tr = pd.concat([y_tr, y_or], axis=0, ignore_index=True)

    print("Base fold shapes:", X_tr.shape, X_val.shape, X_tst.shape)

    te_seed = CFG.SEED + fold * 100
    fold_ids = make_inner_fold_ids(
        y_tr,
        n_splits=CFG.TE_FOLDS,
        seed=te_seed,
    )

    # ------------------------
    # Frequency Encoding
    # ------------------------
    for c in FE_SINGLE_COLS:
        if c in X_tr.columns:
            add_frequency_encode(
                X_tr,
                X_val,
                X_tst,
                cols=(c,),
                out_col=f"fe__{safe_colname(c)}",
                normalize=True,
            )

    for cols in SELECTED_PAIR_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_frequency_encode(
                X_tr,
                X_val,
                X_tst,
                cols=cols,
                out_col="fe2__" + "__".join([safe_colname(c) for c in cols]),
                normalize=True,
            )

    print("After FE:", X_tr.shape, X_val.shape, X_tst.shape)

    # ------------------------
    # Target Encoding
    # ------------------------
    for c in TE_SINGLE_COLS:
        if c in X_tr.columns:
            add_target_encode(
                X_tr,
                X_val,
                X_tst,
                y_tr,
                cols=(c,),
                out_col=f"te__{safe_colname(c)}",
                fold_ids=fold_ids,
                seed=te_seed,
            )

    for cols in SELECTED_PAIR_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_target_encode(
                X_tr,
                X_val,
                X_tst,
                y_tr,
                cols=cols,
                out_col="te2__" + "__".join([safe_colname(c) for c in cols]),
                fold_ids=fold_ids,
                seed=te_seed,
            )

    te_stat_cols = add_te_rowwise_stats(X_tr, X_val, X_tst)

    print("After TE:", X_tr.shape, X_val.shape, X_tst.shape)
    print("TE stat cols:", te_stat_cols)

    # ------------------------
    # Prepare for RealMLP
    # ------------------------
    X_tr_r, X_val_r, X_tst_r = prepare_realmlp_frames(
        X_tr,
        X_val,
        X_tst,
        cat_like_cols=CAT_LIKE_COLS,
    )

    print("RealMLP input shapes:", X_tr_r.shape, X_val_r.shape, X_tst_r.shape)

    y_tr_np = y_tr.values.astype(np.int64)
    y_val_np = y_val.values.astype(np.int64)

    # ------------------------
    # Train configs
    # ------------------------
    for cfg_i, cfg in enumerate(REALMLP_CONFIGS):
        cfg_name = cfg["name"]
        seed = CFG.SEED + fold * 1000 + cfg_i * 100

        print("\n" + "-" * 80)
        print(f"Config: {cfg_name}")
        print("-" * 80)

        val_pred, tst_pred = train_one_realmlp(
            X_tr=X_tr_r,
            y_tr=y_tr_np,
            X_val=X_val_r,
            y_val=y_val_np,
            X_tst=X_tst_r,
            cfg=cfg,
            seed=seed,
        )

        oof_dict[cfg_name][val_idx] = val_pred
        test_dict[cfg_name] += tst_pred / CFG.FOLDS

        fold_auc = roc_auc_score(y_val_np, val_pred)

        print(f"Fold {fold} | {cfg_name} | AUC: {fold_auc:.6f}")

        fold_scores.append({
            "fold": fold,
            "config": cfg_name,
            "auc": fold_auc,
        })

    del X_tr, X_val, X_tst
    del X_tr_r, X_val_r, X_tst_r
    del y_tr, y_val, y_tr_np, y_val_np, fold_ids
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Evaluation / Save
# ============================================================

summary = []

for cfg in REALMLP_CONFIGS:
    cfg_name = cfg["name"]

    oof_pred = oof_dict[cfg_name]
    tst_pred = test_dict[cfg_name]

    auc = roc_auc_score(y, oof_pred)

    print("\n" + "=" * 80)
    print(f"{cfg_name} OOF AUC: {auc:.6f}")
    print("=" * 80)

    oof_path = f"oof_{CFG.OUTPUT_PREFIX}_{cfg_name}_{auc:.6f}.csv"
    test_path = f"test_{CFG.OUTPUT_PREFIX}_{cfg_name}_{auc:.6f}.csv"
    sub_path = f"submission_{CFG.OUTPUT_PREFIX}_{cfg_name}_{auc:.6f}.csv"

    pd.DataFrame({
        CFG.ID: train_id,
        CFG.TARGET: oof_pred,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        CFG.ID: test_id,
        CFG.TARGET: tst_pred,
    }).to_csv(test_path, index=False)

    pd.DataFrame({
        CFG.ID: test_id,
        CFG.TARGET: tst_pred,
    }).to_csv(sub_path, index=False)

    summary.append({
        "kind": "single",
        "name": cfg_name,
        "oof_auc": auc,
        "oof_path": oof_path,
        "test_path": test_path,
        "submission_path": sub_path,
    })


# ============================================================
# Simple Equal Blend
# ============================================================

if len(REALMLP_CONFIGS) >= 2:
    blend_name = "blend_equal_" + "__".join([cfg["name"] for cfg in REALMLP_CONFIGS])

    blend_oof = np.mean(
        [oof_dict[cfg["name"]] for cfg in REALMLP_CONFIGS],
        axis=0,
    ).astype(np.float32)

    blend_test = np.mean(
        [test_dict[cfg["name"]] for cfg in REALMLP_CONFIGS],
        axis=0,
    ).astype(np.float32)

    blend_auc = roc_auc_score(y, blend_oof)

    print("\n" + "=" * 80)
    print(f"{blend_name} OOF AUC: {blend_auc:.6f}")
    print("=" * 80)

    oof_path = f"oof_{CFG.OUTPUT_PREFIX}_{blend_name}_{blend_auc:.6f}.csv"
    test_path = f"test_{CFG.OUTPUT_PREFIX}_{blend_name}_{blend_auc:.6f}.csv"
    sub_path = f"submission_{CFG.OUTPUT_PREFIX}_{blend_name}_{blend_auc:.6f}.csv"

    pd.DataFrame({
        CFG.ID: train_id,
        CFG.TARGET: blend_oof,
    }).to_csv(oof_path, index=False)

    pd.DataFrame({
        CFG.ID: test_id,
        CFG.TARGET: blend_test,
    }).to_csv(test_path, index=False)

    pd.DataFrame({
        CFG.ID: test_id,
        CFG.TARGET: blend_test,
    }).to_csv(sub_path, index=False)

    summary.append({
        "kind": "blend",
        "name": blend_name,
        "oof_auc": blend_auc,
        "oof_path": oof_path,
        "test_path": test_path,
        "submission_path": sub_path,
    })


summary_df = pd.DataFrame(summary).sort_values("oof_auc", ascending=False)
summary_df.to_csv(f"{CFG.OUTPUT_PREFIX}_summary.csv", index=False)

fold_scores_df = pd.DataFrame(fold_scores)
fold_scores_df.to_csv(f"{CFG.OUTPUT_PREFIX}_fold_scores.csv", index=False)

display(summary_df)
display(fold_scores_df)